# Document Stores & MongoDB Basics

## What you will learn in this course 🧐🧐

MongoDB stores data as flexible, JSON-like documents (BSON). In this lecture, you’ll build a solid mental model of document databases, learn the core query primitives, and get hands-on with CRUD, filtering, projections, and pagination. We’ll also compare the document model with relational tables so you can choose the right tool for each job.

In this lecture, you will:

* understand collections, documents, and the role of `_id`/ObjectId,
* load data and perform **CRUD** (create, read, update, delete),
* write **filters** with operators (`$gt`, `$in`, `$and`, regex, array queries),
* use **projections**, **sorting**, **limit/skip** for pagination,
* compare **Mongo queries** with SQL equivalents,
* avoid common pitfalls (full scans, unbounded results, missing projections).

## Why Document Stores?

Relational databases shine with stable schemas and normalized relationships. Document stores trade rigid schemas for **flexibility**, **nested structures**, and **fewer joins**. This is ideal when your records are naturally hierarchical (e.g., a movie with cast, awards, ratings) or when schemas evolve frequently during product iteration.

### Document mental model vs Relational mental model

When you work with document stores, a handful of terms keep popping up - here are the key ones alongside their closest SQL equivalents:

| Document store concept                      | Relational database equivalent |
| ------------------------------------------- | ------------------------------ |
| **Database**                                | Database                       |
| **Collection**                              | **Table**                      |
| **Document** (JSON/BSON)                    | **Row**                        |
| **Fields**                                  | **Columns**                    |
| **`_id`** (auto ObjectId if not provided)   | **Primary key**                |


<Note type="tip" title="When MongoDB fits well">

- You want to store **nested** data without complex joins.
- The schema **changes often**.
- You read a parent and its small children **together** most of the time.

</Note>

## Getting Ready

Let's get our hands dirty and play around with a MongoDB cluster. To do so:

1. Go to [MongoDB official website](https://www.mongodb.com/) 
2. Create a free account by clicking on *Get Started*
3. Once you sign up, go to the *Clusters* section. It should look like this:

![](https://full-stack-assets.s3.eu-west-3.amazonaws.com/MongoDB-1.png)

4. Build a Cluster 
    👉 Choose **Free** cluster
    👉 Give your cluster a name (or leave the default `Cluster0`)
    👉 Choose **AWS** as Cloud provider 
    👉 Choose the closest region you are living in (i.e `eu-west-3`)
    👉 Click on *Create Deployment*

<Note type="note" title="Connect to cluster pop up">

Once you deployed your cluster, a helper pop up window like the one below should show up:

![](https://full-stack-assets.s3.eu-west-3.amazonaws.com/MongoDB-2.png)

It is here to help you connect using all the different methods MongoDB offers. You can explore them as much as you like. During the course we will use the Python driver. 

</Note>

5. To be able to access the DB, you will need to whitelist your public IP address. **This should be done automatically** but if it is not the case:
    👉 Go the *Network Access* tab 
    👉 Click on *Add IP Address* and provide your public IP

<Note type="note" title="What is a public IP and why do I need to whitelist it?">

A public IP is the address your internet provider exposes to the outside world. It is how services on the web identify the device (or router) you are coming from. You can check it with tools like https://whatismyipaddress.com/. 

MongoDB Atlas blocks every connection by default, so adding your public IP to the allowlist tells Atlas, *requests from this address are safe, let them through.* If your provider rotates IPs, remember to update the allowlist when the address changes.

</Note>

6. Now you also need a **user** to access the DB. By default an admin account is created on your cluster and a password is provided to you. If that is not the case (or if you forgot your password 😉)
    👉 Go the *Database Access* tab   
    👉 Either click on *ADD NEW DATABASE USER* or click on the current user to reset your password

![](https://full-stack-assets.s3.eu-west-3.amazonaws.com/MongoDB-3.png)

7. Now you should be able to access the DB
    👉 Back in the *Clusters* tab, click on *Connect*
    👉 Choose Drivers > Python 
    👉 copy the `mongodb+srv://...` URI.

![](https://full-stack-assets.s3.eu-west-3.amazonaws.com/MongoDB-4.png)


## Connect from Python (PyMongo)

Install the official driver before opening your notebook or script:

In [36]:
!pip install pymongo

Let's now initialize the whole database

In [ ]:
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

USERNAME = "xxxxxx_db_user" # Replace with your username 
PASSWORD = "xxxxxx_db_password" # Replace with your password
CLUSTER_NAME= "Cluster0" # Replace with your cluster name
MONGODB_URI=f"mongodb+srv://{USERNAME}:{PASSWORD}@{CLUSTER_NAME.lower()}.1tsvfmh.mongodb.net/?retryWrites=true&w=majority&appName={CLUSTER_NAME.lower()}"

client = MongoClient(MONGODB_URI, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


Everything works fine! 🎉 

## Explore `sample_analytics` with PyMongo

<Note type="important" title="Load the sample dataset">

For the following demo to work, load the MongoDB sample data in Atlas:

👉 Open your cluster and click *...* > **Load Sample Dataset**  
👉 Choose `sample_analytics`

![](https://full-stack-assets.s3.eu-west-3.amazonaws.com/MongoDB-5.png)

</Note>

Let's now connect to the `sample_analytics` database. Atlas ships it with three collections: 

* `customers` 
* `accounts` 
* `transactions` 

The snippets below interact with those collections:

In [38]:
db = client["sample_analytics"]
customers = db.customers
accounts = db.accounts
transactions = db.transactions

print(customers.estimated_document_count()) 

503


## Read data

Let's first query all three collections to see what is inside:

In [51]:
from pprint import pprint

# Helper function to display samples
def show_samples(cursor, title=None):
    if title:
        print(f"\n{title}\n" + "-" * len(title))
    for doc in cursor:
        pprint(doc)

# Display samples from each collection
show_samples(
    customers.find({}, {"_id": 0, "name": 1, "address": 1}).limit(1),
    "Sample customers"
)

show_samples(
    accounts.find({}, {"_id": 0, "account_id": 1, "limit": 1, "products": 1}).limit(1),
    "Sample accounts"
)

show_samples(
    transactions.find({}, {"_id": 0, "account_id": 1, "transactions": {"$slice": 1}}).limit(1),
    "Sample transactions (first entry per account)"
)



Sample customers
----------------
{'address': '38456 Rachael Causeway Apt. 735\nEvanfort, AR 33893',
 'name': 'John Parks'}

Sample accounts
---------------
{'account_id': 557378,
 'limit': 10000,
 'products': ['InvestmentStock', 'Commodity', 'Brokerage', 'CurrencyService']}

Sample transactions (first entry per account)
---------------------------------------------
{'account_id': 443178,
 'transactions': [{'amount': 7514,
                   'date': datetime.datetime(2003, 9, 9, 0, 0),
                   'price': '19.1072802650074180519368383102118968963623046875',
                   'symbol': 'adbe',
                   'total': '143572.1039112657392422534031',
                   'transaction_code': 'buy'}]}


<Note type="important" title="Projection matters">
Always project only the fields you need. Pulling entire documents forces MongoDB to load large arrays/embedded docs and ship them over the wire. A tight projection such as:

```python
accounts.find({"limit": {"$gt": 25000}}, {"_id": 0, "account_id": 1})
```

results in smaller payloads, covered index reads, and faster notebooks.
</Note>


## Create (Insert)

Go on the MongoDB GUI:

👉 Clusters > Browse Collections 
👉 Go to your `customers`

Browse a little, you will see that a sample document looks like this:

```json
{
  "_id": { "$oid": "5ca4bbcea2dd94ee58162a68" },
  "username": "fmiller",
  "name": "Elizabeth Ray",
  "address": "9286 Bethany Glens\nVasqueztown, CO 22939",
  "birthdate": { "$date": { "$numberLong": "226117231000" } },
  "email": "arroyocolton@gmail.com",
  "active": true,
  "accounts": [
    { "$numberInt": "371138" },
    { "$numberInt": "324287" },
    { "$numberInt": "276528" },
    { "$numberInt": "332179" },
    { "$numberInt": "422649" },
    { "$numberInt": "387979" }
  ],
  "tier_and_details": {
    "0df078f33aa74a2e9696e0520c1a828a": {
      "tier": "Bronze",
      "id": "0df078f33aa74a2e9696e0520c1a828a",
      "active": true,
      "benefits": ["sports tickets"]
    },
    "699456451cc24f028d2aa99d7534c219": {
      "tier": "Bronze",
      "benefits": ["24 hour dedicated line", "concierge services"],
      "active": true,
      "id": "699456451cc24f028d2aa99d7534c219"
    }
  }
}

```

Let's try to insert new data inside this collection: 

In [52]:
from datetime import datetime, timezone
import uuid # for generating unique IDs

doc = {
    # "_id": ObjectId(),  # optional - will be created automatically if not provided
    "username": "hs0lo",
    "name": "Han Solo",
    "address": "Docking Bay 94\nMos Eisley, Tatooine 00094",
    "birthdate": datetime(1977, 5, 25, 0, 0, 0, tzinfo=timezone.utc), # MongoDB stores dates in UTC so specify timezone
    "email": "han.solo@falcon.space",
    "active": True,
    "accounts": [1138, 1977, 1211, 7, 94],  
    "tier_and_details": {
        uuid.uuid4().hex: {
            "tier": "Kessel Gold",
            "id": uuid.uuid4().hex,
            "active": True,
            "benefits": [
                "priority docking at Mos Eisley",
                "wookiee co-pilot coverage",
                "hyperspace roadside assistance"
            ],
        },
        uuid.uuid4().hex: {
            "tier": "Rebel Platinum",
            "id": uuid.uuid4().hex, # This is will look like '9b1deb4d-3b7d-4bad-9bdd-2b0d7b3dcb6d'
            "active": True,
            "benefits": [
                "24/7 rebel alliance hotline",
                "beskar upgrade vouchers",
                "imperial entanglement waiver"
            ],
        },
    },
}

# Insert the document
result = db.customers.insert_one(doc)
print("Inserted _id:", result.inserted_id)

Inserted _id: 69b3e024661b5e19597ee532


Perfect! As you can see we now have a new customer in our database:

In [55]:
target_email = "han.solo@falcon.space"
name = "Han Solo"

customers.find_one(
    {"name": name}, 
    {"_id": 0, "username": 1, "name": 1, "address": 1}
)

{'username': 'hs0lo',
 'name': 'Han Solo',
 'address': 'Docking Bay 94\nMos Eisley, Tatooine 00094'}

Now the beauty with Document Store is its **flexibility**. You can insert data that doesn't have to exactly follow the exact same data structure as other documents:

In [57]:
from datetime import datetime, timezone

outer_rim_profile = {
    "username": "lars.archive",
    "display_name": "Moisture Ops",
    "contact": {
        "primary_email": "binary_sunset@tat.cool",
        "holonet_handle": "@blue-milk"
    },
    "is_active": True,
    "memberships": [
        {
            "guild": "Lars Homestead Cooperative",
            "role": "Logistics",
            "joined_at": datetime(2023, 5, 4, tzinfo=timezone.utc)
        }
    ],
    "favorite_droids": [
        {"model": "R2 unit", "serial": "R2-D2"},
        {"model": "Astromech", "serial": "R5-D4"}
    ],
    "inventory": {
        "vaporators": 7,
        "landspeeder": {"make": "SoroSuub", "status": "needs power converter"},
        "moisture_yield_per_cycle": 42.7
    },
    "last_binary_sunset": datetime.now(timezone.utc),
    "notes": [
        "Twin suns still brutal at midday.",
        "Ask nephew to pick up power converters next time in Anchorhead."
    ]
}

result = customers.insert_one(outer_rim_profile)
print("Inserted customer_id:", result.inserted_id)

# Find the newly inserted document

customer = customers.find_one(
    {"username": "lars.archive"}, {"_id": 0, "username": 1, "display_name": 1, "contact": 1, "inventory": 1}
)

pprint(f"Newly inserted customer profile:\n{customer}")

Inserted customer_id: 69b3e3b3661b5e19597ee534
('Newly inserted customer profile:\n'
 "{'username': 'lars.archive', 'display_name': 'Moisture Ops', 'contact': "
 "{'primary_email': 'binary_sunset@tat.cool', 'holonet_handle': '@blue-milk'}, "
 "'inventory': {'vaporators': 7, 'landspeeder': {'make': 'SoroSuub', 'status': "
 "'needs power converter'}, 'moisture_yield_per_cycle': 42.7}}")


<Note type="note" title="Why and when would documents have different data structure?">

MongoDB does not force a rigid schema, so it’s normal to see documents evolving over time. This flexibility kicks in when you add new product features, ingest data from several partners, or support totally different customer types in one place. Instead of blocking on schema migrations, you can accept new fields right away and keep shipping. There a benefits and trade-offs from doing that. Here is a simple view:

| Benefits | Trade-offs |
| --- | --- |
| Iterate quickly without downtime | Application code must handle missing/extra fields |
| Model each record in the shape that best fits the domain | Indexes become trickier to design |
| Store sparse/optional data efficiently | Analytics/export pipelines may need extra cleaning |

A pro tip is if your data structure changes, add a lightweight "type" field or version flag so downstream jobs know which structure to expect.

</Note>



## Update

Of course, you can update data the following way:

In [58]:
result = customers.update_one(
    {"username": "hs0lo"},
    {"$set": {"email": "han.solo@rebellion.space"}},
    upsert=False  # Set it to True to insert if not found
)

print(f"Matched: {result.matched_count}, modified: {result.modified_count}")

Matched: 1, modified: 0


## Delete

And finally, you can delete the following way:

In [59]:
delete_result = customers.delete_one({"username": "hs0lo"})
print(f"Deleted documents: {delete_result.deleted_count}")

Deleted documents: 1


## Filters & Operators

When you build a `find()` query you combine small filter operators. Here are the families you will use all the time:

### Comparison operators

* **`$eq`** 👉 equal to `{ "status": { "$eq": "active" } }`
* **`$ne`** 👉 not equal `{ "state": { "$ne": "CA" } }`
* **`$gt`** 👉 greater than `{ "balance": { "$gt": 1000 } }`
* **`$gte`** 👉 greater than or equal `{ "created_at": { "$gte": ISODate("2025-10-01") } }`
* **`$lt`** 👉 less than `{ "qty": { "$lt": 10 } }`
* **`$lte`** 👉 less than or equal `{ "score": { "$lte": 50 } }`

### Logical operators

* **`$and`** 👉 all conditions must be true `{ "$and": [ { "active": true }, { "tier": "Gold" } ] }`
* **`$or`** 👉 at least one condition must be true `{ "$or": [ { "country": "FR" }, { "country": "DE" } ] }`
* **`$not`** 👉 negate a single condition `{ "age": { "$not": { "$gte": 18 } } } // younger than 18`
* **`$nor`** 👉 none of the listed conditions may be true `{ "$nor": [ { "vip": true }, { "banned": true } ] }`

### Membership operators

* **`$in`** 👉 value is in list `{ "address.state": { "$in": ["CA", "NY"] } }`
* **`$nin`** 👉 value is not in list `{ "country": { "$nin": ["US", "CA"] } }`

### Regex

* **`$regex`** 👉 match text with a regular expression `{ "email": { "$regex": "@example\\.com$", "$options": "i" } } // ends with @example.com, case-insensitive`
* **`$options`** 👉 regex flags (`"i"` case-insensitive, `"m"` multiline, etc.)

### Array operators

* **`$size`** 👉 array has exact length `{ "accounts": { "$size": 2 } }`
* **`$all`** 👉 array contains all listed values (order doesn’t matter) `{ "products": { "$all": ["Investment", "Brokerage"] } }`
* **`$elemMatch`** 👉 at least one array element matches full subquery `{ "transactions": { "$elemMatch": { "amount": { "$lt": -200 }, "category": "Groceries" }}}`

Mix these building blocks to express nearly any MongoDB filter you need. Let's write a basic example:

In [66]:
# Match any account that has at least one matching sub-transaction
query = {
    "transactions": {
        "$elemMatch": { # Match at least one transaction in the transactions array
            "transaction_code": "buy",
            "symbol": "amd",
            "amount": {"$gte": 300} # Will query for the amount of stock bought in one transaction greater than 8000
        }
    }
}

# Positional $ returns only the first matching array element
projection = {
    "_id": 0,
    "account_id": 1,
    "transactions.$": 1
}

doc = transactions.find_one(query, projection)
pprint(doc)

{'account_id': 716662,
 'transactions': [{'amount': 8592,
                   'date': datetime.datetime(2008, 3, 19, 0, 0),
                   'price': '6.25868566899633460565155473886989057064056396484375',
                   'symbol': 'amd',
                   'total': '53774.62726801650693175815832',
                   'transaction_code': 'buy'}]}


## SQL vs PyMongo quick equivalence

To help you memorize the difference methods that `pymongo` offers, here is a quick equivalence with classic SQL queries:

| SQL                                                                                     | PyMongo                                                                                                                                          |
| --------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------ |
| `SELECT username, email FROM customers WHERE address_state = 'CA';`                     | `list(customers.find({"address.state": "CA"}, {"_id": 0, "username": 1, "email": 1}))`                                                           |
| `SELECT account_id, limit FROM accounts ORDER BY limit DESC LIMIT 5;`                   | `list(accounts.find({}, {"_id": 0, "account_id": 1, "limit": 1}).sort("limit", -1).limit(5))`                                                    |
| `UPDATE customers SET vip = TRUE WHERE birth_year < 1980;`                              | `customers.update_many({"birthdate": {"$lt": datetime(1980, 1, 1, tzinfo=timezone.utc)}}, {"$set": {"vip": True}})`                              |
| `DELETE FROM customers WHERE username LIKE 'training_%';`                               | `customers.delete_many({"username": {"$regex": "^training_"}})`                                                                                  |
| `SELECT * FROM transactions WHERE amount < -200 AND category = 'Groceries';`            | `list(transactions.find({"transactions": {"$elemMatch": {"amount": {"$lt": -200}, "category": "Groceries"}}}))`                                  |

## Working with Dates & ObjectIds

Let's finish the lecture with two concepts that you need to pay attention to with `Dates` & `ObjectIds`

* **Dates (Python `datetime` ↔ BSON `Date`)** With PyMongo, datetimes are stored in UTC as BSON `Date`. The gotchas are:

  * **Timezone awareness:** PyMongo can treat datetimes as naïve (default) or timezone-aware (UTC) depending on your settings. Mixing naïve and aware values leads to subtle bugs (wrong comparisons, off-by-hours filters, failed equality checks).
  * **Consistency across your stack:** If your app or notebook constructs naïve `datetime` objects while your DB stores UTC, the same query can behave differently on different machines.
  * **Indexing & performance:** Range queries on dates only hit indexes if the stored type is a real `Date`. Storing timestamps as strings breaks sort/order semantics and index use.
    In short: you must decide and **standardize** on timezone-aware UTC handling in PyMongo (e.g., via client/codec options) to avoid incorrect filters, joins, and reporting.

* **ObjectIds (Python `bson.ObjectId` ↔ `_id`)** MongoDB’s default `_id` is a **BSON ObjectId** (a 12-byte binary type that embeds a creation timestamp). The pitfalls are:

  * **Type matching:** Filtering by `_id` with a **string** won’t match an ObjectId-typed field. In PyMongo you must use `bson.ObjectId`; otherwise the query silently returns nothing and bypasses indexes.
  * **Sorting & pagination semantics:** Because ObjectIds encode creation time, they’re commonly used for stable ordering and seek-based pagination. Treating them as strings destroys that property and can yield inconsistent pages.
  * **Diagnostics & interoperability:** ObjectIds round-trip to Python as a dedicated type; converting them ad-hoc to strings in some parts of the code (but not others) creates hard-to-trace bugs and broken links between documents.

That is why you should always **make dates explicitly UTC and consistently timezone-aware**, and **treat `_id` as an ObjectId (not a string)**. Getting these two types right prevents empty-result “mystery” queries, preserves index performance, and keeps pagination and time-based logic reliable.

<Note type="note" title="What is bson?">

BSON (“Binary JSON”) is MongoDB’s compact, binary-encoded format for storing and transmitting documents. It keeps JSON’s familiar structure (objects, arrays) but adds rich types that JSON lacks—like ObjectId, Date, 64-bit integers, Decimal128, binary data, and regex—plus ordered fields and length prefixes for fast scanning. BSON is what MongoDB uses on disk and over the wire, enabling efficient storage, indexing, and precise numeric handling while remaining easy to map to JSON in apps and tools.

</Note>



In [67]:
from datetime import datetime, timezone
from bson import ObjectId

# Transactions recorded since 1 Jan 2020 (UTC)
start = datetime(2008, 1, 1, tzinfo=timezone.utc)
list(
    transactions.find(
        {"transactions.date": {"$gte": start}},
        {"_id": 0, "account_id": 1, "transactions": {"$slice": -1}}
    )
)

# Convert a string to ObjectId before querying (paste a real _id from your data)
#customer_id = ObjectId("69b307d7661b5e19597ee52f")
#customers.find_one({"_id": customer_id})

[{'account_id': 443178,
  'transactions': [{'date': datetime.datetime(2005, 7, 7, 0, 0),
    'amount': 2881,
    'transaction_code': 'buy',
    'symbol': 'msft',
    'price': '20.6769287918292690164889791049063205718994140625',
    'total': '59570.23184926012403650474880'}]},
 {'account_id': 278603,
  'transactions': [{'date': datetime.datetime(2016, 5, 3, 0, 0),
    'amount': 6545,
    'transaction_code': 'buy',
    'symbol': 'goog',
    'price': '695.0931497813386386042111553251743316650390625',
    'total': '4549384.665318861389664562012'}]},
 {'account_id': 674364,
  'transactions': [{'date': datetime.datetime(2016, 6, 13, 0, 0),
    'amount': 4058,
    'transaction_code': 'buy',
    'symbol': 'goog',
    'price': '718.7567657609789648631704039871692657470703125',
    'total': '2916714.955458052639414745499'}]},
 {'account_id': 126668,
  'transactions': [{'date': datetime.datetime(2015, 10, 26, 0, 0),
    'amount': 8117,
    'transaction_code': 'buy',
    'symbol': 'goog',
    'pri

<Note type="important" title="Timezones">
Store UTC timestamps (`datetime(..., tzinfo=timezone.utc)`). Convert to local time in your application or BI layer.
</Note>

## Resources 📚📚

* [MongoDB Docs - Getting Started](https://www.mongodb.com/docs/manual/introduction/)
* [MongoDB Docs - CRUD Operations](https://www.mongodb.com/docs/manual/crud/)
* [MongoDB Docs - Query and Projection Operators](https://www.mongodb.com/docs/manual/reference/operator/query/)
* [MongoDB Schema Design Best Practices](https://www.mongodb.com/developer/products/mongodb/schema-design-best-practices/)
* [MongoDB Docs - Aggregation Framework](https://www.mongodb.com/docs/manual/aggregation/)
* [MongoDB Docs - Indexes](https://www.mongodb.com/docs/manual/indexes/)
* [MongoDB Atlas](https://www.mongodb.com/atlas/database)
* [PyMongo Documentation](https://pymongo.readthedocs.io/)
* [MongoDB Compass](https://www.mongodb.com/products/compass)
* [MongoDB University](https://learn.mongodb.com/)
* [MongoDB Atlas Sample Datasets](https://www.mongodb.com/docs/atlas/sample-data/)